In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline
from pathlib import Path

OUTPUTS_DIR = Path("../Outputs")
LOCKED_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

df = pd.read_csv(OUTPUTS_DIR / "merged_text_905.csv")
print(f"Posts with usable text: {df['has_text'].sum()}")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Posts with usable text: 797


In [2]:
MODELS_TO_TEST = [
    {
        "name": "michellejieli",
        "hf_path": "michellejieli/emotion_text_classifier",
        "output_file": "emotion_michellejieli_905.csv",
    },
    {
        "name": "bhadresh_distil",
        "hf_path": "bhadresh-savani/distilbert-base-uncased-emotion",
        "output_file": "emotion_bhadresh_distil_905.csv",
    },
    {
        "name": "hartmann_large",
        "hf_path": "j-hartmann/emotion-english-roberta-large",
        "output_file": "emotion_hartmann_large_905.csv",
    },
    {
        "name": "emoRoBERTa",
        "hf_path": "arpanghoshal/EmoRoBERTa",
        "output_file": "emotion_emoroberta_905.csv",
    },
]

# Mapping from each model's label set to our locked 6 emotions
# Fill this in AFTER seeing what labels each model outputs
LABEL_MAPPINGS = {
    "michellejieli": None,  # will fill after inspecting
    "bhadresh_distil": None,
    "hartmann_large": None,
    "emoRoBERTa": None,
}

In [3]:
for model_config in MODELS_TO_TEST:
    print(f"\n=== {model_config['name']} ({model_config['hf_path']}) ===")
    try:
        model = pipeline(
            "text-classification",
            model=model_config["hf_path"],
            top_k=None,
            truncation=True
        )
        test = model("I'm so excited for the Super Bowl!")
        labels = [item["label"] for item in test[0]]
        print(f"Labels ({len(labels)}): {labels}")
    except Exception as e:
        print(f"FAILED to load: {e}")
    finally:
        # Free memory before next model
        del model
        import gc
        gc.collect()


=== michellejieli (michellejieli/emotion_text_classifier) ===


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2294.78it/s]


Labels (7): ['joy', 'neutral', 'surprise', 'anger', 'fear', 'sadness', 'disgust']

=== bhadresh_distil (bhadresh-savani/distilbert-base-uncased-emotion) ===


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1263.90it/s]


Labels (6): ['joy', 'love', 'anger', 'fear', 'surprise', 'sadness']

=== hartmann_large (j-hartmann/emotion-english-roberta-large) ===


/usr/local/python/3.12.1/lib/python3.12/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 1421.52 MB. The target location /home/codespace/.cache/huggingface/hub/models--j-hartmann--emotion-english-roberta-large/blobs only has 971.47 MB free disk space.
  warnings.warn(
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 13159.63it/s]
/usr/local/python/3.12.1/lib/python3.12/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.00 MB. The target location /home/codespace/.cache/huggingface/hub/models--j-hartmann--emotion-english-roberta-large/blobs only has 0.00 MB free disk space.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.80 MB. The target location /ho

Labels (7): ['joy', 'surprise', 'neutral', 'sadness', 'anger', 'fear', 'disgust']

=== emoRoBERTa (arpanghoshal/EmoRoBERTa) ===
FAILED to load: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/arpanghoshal/EmoRoBERTa.
401 Client Error. (Request ID: Root=1-6a48337f-5be2303a0a78db914a14d03d;8714d1a7-871f-486e-ba87-a15feba49c62)

Cannot access gated repo for url https://huggingface.co/arpanghoshal/EmoRoBERTa/resolve/main/config.json.
Access to model arpanghoshal/EmoRoBERTa is restricted. You must have access to it and be authenticated to access it. Please log in.


NameError: name 'model' is not defined

In [4]:
import gc

# Only use the models that loaded successfully
WORKING_MODELS = [
    {
        "name": "michellejieli",
        "hf_path": "michellejieli/emotion_text_classifier",
        "output_file": "emotion_michellejieli_905.csv",
    },
    {
        "name": "bhadresh_distil",
        "hf_path": "bhadresh-savani/distilbert-base-uncased-emotion",
        "output_file": "emotion_bhadresh_distil_905.csv",
    },
]

texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:1500] for t in texts]

all_results = {}
for cfg in WORKING_MODELS:
    print(f"\n=== Running {cfg['name']} on {len(texts)} posts ===")
    model = pipeline(
        "text-classification",
        model=cfg["hf_path"],
        top_k=None,
        truncation=True
    )
    results = model(texts, batch_size=8)
    all_results[cfg["name"]] = results
    print(f"Done. Got {len(results)} results")
    del model
    gc.collect()

print("\nAll models done.")


=== Running michellejieli on 797 posts ===


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15205.67it/s]


Done. Got 797 results

=== Running bhadresh_distil on 797 posts ===


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 218.85it/s]


Done. Got 797 results

All models done.


In [5]:
# Label mappings for each model
LABEL_MAPPINGS = {
    "michellejieli": {
        # 7 labels: joy, neutral, surprise, anger, fear, sadness, disgust
        "anger": ["anger"],
        "disgust": ["disgust"],
        "fear": ["fear"],
        "joy": ["joy"],
        "sadness": ["sadness"],
        "surprise": ["surprise"],
    },
    "bhadresh_distil": {
        # 6 labels: joy, love, anger, fear, surprise, sadness (no disgust, has love)
        "anger": ["anger"],
        "disgust": [],  # not in this model
        "fear": ["fear"],
        "joy": ["joy", "love"],  # merge love into joy
        "sadness": ["sadness"],
        "surprise": ["surprise"],
    },
}

for cfg in WORKING_MODELS:
    name = cfg["name"]
    results = all_results[name]
    mapping = LABEL_MAPPINGS[name]
    
    # Build a dataframe of all scores per post
    all_scores = pd.DataFrame([
        {item["label"]: item["score"] for item in r} for r in results
    ])
    
    # Collapse into our locked 6 emotions using MAX (generous mapping)
    mapped = pd.DataFrame()
    for our_emo, source_labels in mapping.items():
        valid = [c for c in source_labels if c in all_scores.columns]
        if valid:
            mapped[f"emo_{our_emo}"] = all_scores[valid].max(axis=1)
        else:
            mapped[f"emo_{our_emo}"] = 0.0
    
    # Attach to df
    mapped.index = df.index[df["has_text"]]
    out_df = df.copy()
    # Drop existing columns to avoid conflicts
    out_df = out_df.drop(columns=[c for c in mapped.columns if c in out_df.columns])
    out_df = out_df.join(mapped)
    
    # Dominant emotion
    emo_cols = [f"emo_{e}" for e in LOCKED_EMOTIONS]
    out_df["dominant_emotion"] = None
    out_df["dominant_emotion_score"] = None
    mask = out_df["has_text"]
    out_df.loc[mask, "dominant_emotion"] = out_df.loc[mask, emo_cols].idxmax(axis=1).str.replace("emo_", "")
    out_df.loc[mask, "dominant_emotion_score"] = out_df.loc[mask, emo_cols].max(axis=1)
    
    # Save
    out_path = OUTPUTS_DIR / cfg["output_file"]
    out_df.to_csv(out_path, index=False)
    print(f"\n{name} saved: {out_path}")
    print(f"  Dominant emotion counts:")
    print(out_df["dominant_emotion"].value_counts(dropna=False).to_string())


michellejieli saved: ../Outputs/emotion_michellejieli_905.csv
  Dominant emotion counts:
dominant_emotion
joy         438
None        108
fear        103
surprise     99
anger        73
sadness      60
disgust      24

bhadresh_distil saved: ../Outputs/emotion_bhadresh_distil_905.csv
  Dominant emotion counts:
dominant_emotion
joy         520
anger       164
None        108
fear         66
sadness      39
surprise      8


In [6]:
# Load everything for comparison
master = pd.read_csv(OUTPUTS_DIR / "routed_master_905.csv")
llm_emo_obj = pd.read_csv(OUTPUTS_DIR / "llm_emotion_objective_905.csv")
bertweet = pd.read_csv(OUTPUTS_DIR / "emotion_bertweet_905.csv")

# Load the two new ones
michelle = pd.read_csv(OUTPUTS_DIR / "emotion_michellejieli_905.csv")
bhadresh = pd.read_csv(OUTPUTS_DIR / "emotion_bhadresh_distil_905.csv")

# Yellow bucket posts
yellow_mask = master["emotion_bucket"] == "yellow"
yellow_indices = master.index[yellow_mask]

# Get dominant emotion for each source
def dominant(row, emo_cols):
    scores = {c.replace("emo_", "").replace("cardiff_", "").replace("distil_", "").replace("goemo_", "").replace("llm_", ""): row[c] for c in emo_cols}
    return max(scores, key=scores.get)

# Extract dominant for each model on yellow posts
def get_doms(df_source, prefix=""):
    if prefix:
        cols = [f"{prefix}_{e}" for e in LOCKED_EMOTIONS]
    else:
        cols = [f"emo_{e}" for e in LOCKED_EMOTIONS]
    return df_source.loc[yellow_indices].apply(lambda r: max({e: r[c] for e, c in zip(LOCKED_EMOTIONS, cols)}, key=lambda k: {e: r[c] for e, c in zip(LOCKED_EMOTIONS, cols)}[k]), axis=1)

cardiff_doms = get_doms(master, "cardiff")
distil_doms = get_doms(master, "distil")
goemo_doms = get_doms(master, "goemo")
bertweet_doms = get_doms(bertweet)
michelle_doms = get_doms(michelle)
bhadresh_doms = get_doms(bhadresh)

# LLM doms
llm_renamed = llm_emo_obj.rename(columns={e: f"llm_{e}" for e in LOCKED_EMOTIONS})
llm_merged = master.loc[yellow_indices].merge(
    llm_renamed[[f"llm_{e}" for e in LOCKED_EMOTIONS] + ["_post_index"]],
    left_index=True,
    right_on="_post_index",
    how="inner"
)
llm_doms = llm_merged.apply(lambda r: max({e: r[f"llm_{e}"] for e in LOCKED_EMOTIONS}, key=lambda k: r[f"llm_{k}"]), axis=1)
llm_doms.index = llm_merged["_post_index"]

# Align all to same index
aligned = pd.DataFrame({
    "cardiff": cardiff_doms,
    "distil": distil_doms,
    "goemo": goemo_doms,
    "bertweet": bertweet_doms,
    "michellejieli": michelle_doms,
    "bhadresh": bhadresh_doms,
    "llm": llm_doms,
}).dropna()

print(f"Yellow posts with all models scored: {len(aligned)}\n")

# Agreement with LLM
print("=== Agreement with LLM (yellow posts) ===")
for model in ["cardiff", "distil", "goemo", "bertweet", "michellejieli", "bhadresh"]:
    agree = (aligned[model] == aligned["llm"]).sum()
    pct = agree / len(aligned) * 100
    print(f"  {model:15s} vs LLM:  {agree:4d}/{len(aligned)} ({pct:.1f}%)")

# Agreement with Cardiff
print("\n=== Agreement with Cardiff ===")
for model in ["distil", "goemo", "bertweet", "michellejieli", "bhadresh"]:
    agree = (aligned[model] == aligned["cardiff"]).sum()
    pct = agree / len(aligned) * 100
    print(f"  {model:15s} vs Cardiff: {agree:4d}/{len(aligned)} ({pct:.1f}%)")

Yellow posts with all models scored: 696

=== Agreement with LLM (yellow posts) ===
  cardiff         vs LLM:   474/696 (68.1%)
  distil          vs LLM:   255/696 (36.6%)
  goemo           vs LLM:   292/696 (42.0%)
  bertweet        vs LLM:   457/696 (65.7%)
  michellejieli   vs LLM:   386/696 (55.5%)
  bhadresh        vs LLM:   410/696 (58.9%)

=== Agreement with Cardiff ===
  distil          vs Cardiff:  289/696 (41.5%)
  goemo           vs Cardiff:  288/696 (41.4%)
  bertweet        vs Cardiff:  476/696 (68.4%)
  michellejieli   vs Cardiff:  425/696 (61.1%)
  bhadresh        vs Cardiff:  455/696 (65.4%)
